In [1]:
import os, csv, time, random
from datetime import datetime
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc,
    accuracy_score,roc_auc_score, precision_score, recall_score, f1_score
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Enable GPU memory growth
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs available:", tf.config.list_physical_devices("GPU"))

# # Enable mixed precision
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy("mixed_float16")
# print("Mixed precision enabled:", mixed_precision.global_policy())

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["Normal", "Abnormal"]
NUM_CLASSES = 2

BACKBONE_NAME = "ResNet50"
PRETRAINED = True
# INPUT_SIZE = (224, 224)   # change to (299, 299) later if needed
INPUT_SIZE = (299,299)

# Experiment logging
EXPERIMENT_ROOT = "experiments/binary_resnet50_roc_auc"
# CSV_FILE = os.path.join(EXPERIMENT_ROOT, "experiments.csv")
CSV_FILE = os.path.join(EXPERIMENT_ROOT, "roc_auc_binary_experiments.csv")
# LOSS_PLOT_FOLDER = os.path.join(EXPERIMENT_ROOT, "loss_plots")

CURVE_PLOT_FOLDER = os.path.join(EXPERIMENT_ROOT, "training_curves")
ROC_PLOT_FOLDER = os.path.join(EXPERIMENT_ROOT, "roc_curves")

# os.makedirs(LOSS_PLOT_FOLDER, exist_ok=True)
os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
os.makedirs(CURVE_PLOT_FOLDER, exist_ok=True)
os.makedirs(ROC_PLOT_FOLDER, exist_ok=True)


2026-03-11 21:48:29.945906: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 21:48:29.963374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-11 21:48:29.984425: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-11 21:48:29.991330: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-11 21:48:30.006620: I tensorflow/core/platform/cpu_feature_guar

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1773265712.815317    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773265712.878432    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773265712.880039    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


In [ ]:
# # Create a new, balanced binary dataset, that is a subset of the original tfrecords data

# SEED = 0

# # Parse .tfrecords files
# # Image stored as raw bytes and label stored as int (0, 1, 2)
# feature_description = {
#     "image": tf.io.FixedLenFeature([], tf.string),
#     "label": tf.io.FixedLenFeature([], tf.int64),
# }

# def parse_example(example_proto):
#     example = tf.io.parse_single_example(example_proto, feature_description)
#     image = tf.io.decode_raw(example["image"], tf.uint8)
#     image = tf.reshape(image, [299, 299, 1])
#     image = tf.cast(image, tf.float32) / 255.0  # normalize
#     label = tf.cast(example["label"], tf.int32)
#     return image, label

# # Collect all .tfrecords in the training folder
# train_folder = "Kaggle_Data_recheck/Cleaned_Training_Data/" # second data cleaning
# tfrecord_files = glob.glob(os.path.join(train_folder, "*.tfrecords"))

# # Load dataset from all files
# raw_train_ds = tf.data.TFRecordDataset(tfrecord_files)
# raw_train_ds = raw_train_ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)

# # Inspect samples
# for img, lbl in raw_train_ds.take(3):
#     print("Image shape:", img.shape, "Label:", lbl.numpy())

# # Load entire dataset into memory
# images = []
# labels = []

# for img, lbl in raw_train_ds:
#     images.append(img.numpy())
#     labels.append(lbl.numpy())

# images = np.stack(images)
# labels = np.array(labels)

# print("Total samples:", len(labels))

# # Indices for each class
# idx_0 = np.where(labels == 0)[0]        # Normal
# idx_abnormal = np.where(labels != 0)[0] # All abnormal

# print(f"Class counts before balancing:")
# print(f"  Normal (0): {len(idx_0)}")
# print(f"  Abnormal (1+2): {len(idx_abnormal)}")

# # Sample equal number of Normal samples
# rng = np.random.default_rng(SEED)
# idx_0_bal = rng.choice(idx_0, size=len(idx_abnormal), replace=False)

# balanced_idx = np.concatenate([idx_0_bal, idx_abnormal])
# rng.shuffle(balanced_idx)

# images_bal_bin = images[balanced_idx]
# labels_bal_bin = labels[balanced_idx]  # keep original labels (0, 1, 2)

# print(f"\nBalanced dataset size: {len(labels_bal_bin)}")
# print("Samples per class:", np.bincount(labels_bal_bin))

# # Save arrays
# np.save("images_bal_bin.npy", images_bal_bin)
# np.save("labels_bal_bin.npy", labels_bal_bin)


In [2]:
# Load balanced training data
X_train = np.load("images_bal_bin.npy")
y_train = np.load("labels_bal_bin.npy")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_train dtype:", X_train.dtype)
print("Label distribution:", np.bincount(y_train))


X_train shape: (14578, 299, 299, 1)
y_train shape: (14578,)
X_train dtype: float32
Label distribution: [7289 4014 3275]


In [3]:
# Load validation data
# X_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_data.npy")
# y_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_labels.npy")

# Load test data instead, keep all variables the same for simplicity
X_val = np.load("Kaggle_Data_recheck/Cleaned_Test_Data/test10_data.npy")
y_val = np.load("Kaggle_Data_recheck/Cleaned_Test_Data/test10_labels.npy")

# X_val = np.load("validation_data_norm.npy")
# y_val = np.load("validation_labels_norm.npy")

# The preprocessing function negates the need for and use of prior normalization.
# Loading either dataset produces the same result.

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_val dtype:", X_val.dtype)
print("Val label distribution:", np.bincount(y_val))

X_val shape: (7682, 299, 299, 1)
y_val shape: (7682,)
X_val dtype: uint8
Val label distribution: [6697  607  378]


In [4]:
# Binary label mapping
# 0:0 Normal
# 1,2:1 Abnormal

y_train_bin = (y_train != 0).astype("int32")
y_val_bin   = (y_val   != 0).astype("int32")

print("Train distribution:", np.bincount(y_train_bin))
print("Val distribution:",   np.bincount(y_val_bin))


Train distribution: [7289 7289]
Val distribution: [6697  985]


In [5]:
# Data preprocessing
def preprocess(x, y):
    x = tf.cast(x, tf.float32)
    x = tf.image.resize(x, INPUT_SIZE)
    x = tf.image.grayscale_to_rgb(x)
    x = preprocess_input(x)  # ImageNet normalization
    return x, y

In [6]:
# Data augmentation
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    return image, label

In [7]:
BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000

# Training dataset
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train_bin))
    .shuffle(SHUFFLE_BUFFER, seed=SEED)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)  # ← added
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation dataset
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val_bin))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

I0000 00:00:1773265738.282926    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773265738.285024    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773265738.287443    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773265738.416792    1653 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [8]:
# Check one batch
for images, labels in train_ds.take(1):
    print("Train batch images shape:", images.shape)
    print("Train batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())

for images, labels in val_ds.take(1):
    print("Validation batch images shape:", images.shape)
    print("Validation batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())


Train batch images shape: (32, 299, 299, 3)
Train batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'int32'>
Image min/max: -123.74045 -103.12866


2026-03-11 21:49:08.940435: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Validation batch images shape: (32, 299, 299, 3)
Validation batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'int32'>
Image min/max: -123.68 122.061


2026-03-11 21:49:09.170361: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
# Backbone
backbone = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(*INPUT_SIZE, 3)
)
backbone.trainable = False

# Classifier head
inputs = tf.keras.Input(shape=(*INPUT_SIZE, 3))
x = backbone(inputs, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)  # binary
# outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 299, 299, 3)]     0         
                                                                 
 resnet50 (Functional)       (None, 10, 10, 2048)      23587712  
                                                                 
 global_average_pooling2d (  (None, 2048)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 batch_normalization (Batch  (None, 2048)              8192      
 Normalization)                                                  
                                                                 
 dense (Dense)               (None, 512)               1049088   
                                                                 
 batch_normalization_1 (Bat  (None, 512)               2048  

In [10]:
# Binary focal loss function
def binary_focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        w = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        return -tf.reduce_mean(w * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    return loss

In [11]:
# Compile
# LEARNING_RATE = 1e-3

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )


# Unfreeze last ResNet50 block
# backbone.trainable = True
# for layer in backbone.layers[:-33]:  # freeze all except last conv block
#     layer.trainable = False

#     # Unfreeze conv5_x only
# for layer in backbone.layers:
#     layer.trainable = layer.name.startswith("conv5")

# Unfreeze conv4_x and conv5_x
for layer in backbone.layers:
    layer.trainable = (
        # layer.name.startswith("conv2") or
        layer.name.startswith("conv3") or
        layer.name.startswith("conv4") or
        layer.name.startswith("conv5")
    )

# Focal Loss Function

# def focal_loss_per_class(alpha=None, gamma=2.0):
#     alpha = np.array(alpha if alpha is not None else [1.0]*NUM_CLASSES, dtype=np.float32)
    
#     def loss_fn(y_true, y_pred):
#         y_true_int = tf.cast(y_true, tf.int32)
#         y_pred_soft = tf.nn.softmax(y_pred, axis=-1)
#         y_pred_soft = tf.clip_by_value(y_pred_soft, 1e-7, 1-1e-7)
        
#         # Probability of the true class
#         pt = tf.gather(y_pred_soft, y_true_int[:, None], batch_dims=1)
#         pt = tf.squeeze(pt, axis=1)
        
#         # Class weights per sample
#         alpha_t = tf.gather(alpha, y_true_int)
        
#         # Focal loss
#         loss = -alpha_t * (1 - pt)**gamma * tf.math.log(pt)
#         return tf.reduce_mean(loss)
    
#     return loss_fn

# # Class weights for loss function
# alpha_weights = [1.0, 5.0, 8.0]  # Normal, Benign, Malignant

# # Compile with lower LR for fine-tuning
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     # loss="sparse_categorical_crossentropy",
#     loss=focal_loss_per_class(alpha=alpha_weights, gamma=2.8),
#     metrics=["accuracy"]
# )

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    # loss="binary_crossentropy",
    loss=binary_focal_loss(gamma=3.0, alpha=0.5),
    # metrics=["accuracy"]
    metrics=[AUC(name="roc_auc")] 
)

In [12]:
early_stop = EarlyStopping(
    # monitor="val_loss",
    monitor="val_roc_auc",        # monitor ROC-AUC instead of val_loss
    patience=5,
    restore_best_weights=True,
    mode="max"                    # maximize ROC-AUC
)

lr_reduce = ReduceLROnPlateau(
    # monitor="val_loss",
    monitor="val_roc_auc",        # same here
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    mode="max"                    # maximize ROC-AUC
)

callbacks = [early_stop, lr_reduce]

In [13]:
EPOCHS = 30

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    # class_weight={0: 1.0, 1: 10.0, 2: 50.0},
    callbacks=callbacks,
    verbose=1
)

training_time_sec = time.time() - start_time
print(f"Training time (min): {training_time_sec / 60:.2f}")

Epoch 1/30


2026-03-11 21:49:44.766215: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90700
W0000 00:00:1773265784.847960    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.855782    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.857846    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.867500    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.869309    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.871456    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265784.875130    1763 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
'+ptx85' is not a recognized feature for t

  1/456 [..............................] - ETA: 1:08:32 - loss: 0.2650 - roc_auc: 0.5688

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1773265788.564119    1761 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265788.565417    1761 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265788.567199    1761 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265788.569204    1761 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265788.571003    1761 gpu_timer.cc:114] Skipping the delay kernel, measu

  2/456 [..............................] - ETA: 2:43 - loss: 0.2550 - roc_auc: 0.5733   

...

456/456 [==============================] - ETA: 0s - loss: 0.1397 - roc_auc: 0.7761

W0000 00:00:1773265822.066173    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.067531    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.073453    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.073910    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.074375    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.074874    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.075327    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.075908    1764 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265822.076401    1764 gp

456/456 [==============================] - 61s 115ms/step - loss: 0.1397 - roc_auc: 0.7761 - val_loss: 0.5474 - val_roc_auc: 0.7043 - lr: 1.0000e-04
Epoch 2/30
456/456 [==============================] - ETA: 0s - loss: 0.1038 - roc_auc: 0.8071 

W0000 00:00:1773265873.922171    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.922877    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.923709    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.924832    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.925926    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.927249    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.929364    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.931202    1759 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773265873.933120    1759 gp

456/456 [==============================] - 50s 109ms/step - loss: 0.1038 - roc_auc: 0.8071 - val_loss: 0.7148 - val_roc_auc: 0.7100 - lr: 1.0000e-04
Epoch 3/30
456/456 [==============================] - 50s 110ms/step - loss: 0.0856 - roc_auc: 0.8150 - val_loss: 0.7609 - val_roc_auc: 0.7071 - lr: 1.0000e-04
Epoch 4/30
456/456 [==============================] - 50s 110ms/step - loss: 0.0736 - roc_auc: 0.8209 - val_loss: 0.8788 - val_roc_auc: 0.7163 - lr: 1.0000e-04
Epoch 5/30
456/456 [==============================] - 50s 109ms/step - loss: 0.0654 - roc_auc: 0.8218 - val_loss: 0.8180 - val_roc_auc: 0.7172 - lr: 1.0000e-04
Epoch 6/30
456/456 [==============================] - 50s 109ms/step - loss: 0.0578 - roc_auc: 0.8282 - val_loss: 0.6088 - val_roc_auc: 0.7070 - lr: 1.0000e-04
Epoch 7/30
456/456 [==============================] - 50s 109ms/step - loss: 0.0515 - roc_auc: 0.8309 - val_loss: 0.8457 - val_roc_auc: 0.7154 - lr: 1.0000e-04
Epoch 8/30
456/456 [==============================]

In [14]:
# Validation predictions
val_ds_eval = val_ds.unbatch().batch(32)  # ensures consistent batching for predict
y_pred_probs = model.predict(val_ds_eval).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)

# ROC-AUC
roc_auc = roc_auc_score(y_val_bin, y_pred_probs)
print(f"\nROC-AUC: {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_val_bin, y_pred)
print("Confusion matrix:\n", cm)

# Normalized confusion matrix
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Classification report
report_dict = classification_report(
    y_val_bin,          # use binary labels
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)
print(
    "\nClassification report:\n",
    classification_report(
        y_val_bin,
        y_pred,
        target_names=CLASS_NAMES
    )
)

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")

241/241 [==============================] - 19s 72ms/step

ROC-AUC: 0.7425
Confusion matrix:
 [[3897 2800]
 [ 221  764]]

Normalized confusion matrix:
 [[0.58 0.42]
 [0.22 0.78]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.95      0.58      0.72      6697
    Abnormal       0.21      0.78      0.34       985

    accuracy                           0.61      7682
   macro avg       0.58      0.68      0.53      7682
weighted avg       0.85      0.61      0.67      7682

Accuracy for class 0 (Normal): 0.5819
Accuracy for class 1 (Abnormal): 0.7756


2026-03-11 22:00:58.340540: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


In [15]:
# Auto-increment experiment ID
def get_next_experiment_id(csv_file):
    if not os.path.exists(csv_file):
        return 1
    with open(csv_file, "r") as f:
        return sum(1 for _ in f)

# Extract max L1/L2 across layers
def extract_l1_l2(model):
    l1_vals = []
    l2_vals = []
    for layer in model.layers:
        reg = getattr(layer, "kernel_regularizer", None)
        if reg:
            if hasattr(reg, "l1") and reg.l1 is not None: 
                l1_vals.append(float(tf.keras.backend.get_value(reg.l1)))
            if hasattr(reg, "l2") and reg.l2 is not None:
                l2_vals.append(float(tf.keras.backend.get_value(reg.l2)))
    return (
        max(l1_vals) if l1_vals else 0.0,
        max(l2_vals) if l2_vals else 0.0)

# Extract max dropout rate
def extract_max_dropout(model):
    rates = [float(layer.rate) for layer in model.layers if isinstance(layer, tf.keras.layers.Dropout)]
    return max(rates) if rates else 0.0

# Build experiment row
exp_id = get_next_experiment_id(CSV_FILE)
l1_val, l2_val = extract_l1_l2(model)
dropout_val = extract_max_dropout(model)
learning_rate_val = float(model.optimizer.learning_rate.numpy())
train_loss = history.history["loss"][-1]
val_loss = history.history["val_loss"][-1]
# train_acc = history.history["accuracy"][-1]
# val_acc = history.history["val_accuracy"][-1]

row = {
    "experiment_id": exp_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "task": "binary roc-auc",
    "backbone": BACKBONE_NAME,
    "pretrained": PRETRAINED,
    "input_size": f"{INPUT_SIZE[0]}x{INPUT_SIZE[1]}",
    "batch_size": BATCH_SIZE,
    "shuffle_buffer": SHUFFLE_BUFFER,
    "epochs": len(history.history["loss"]),
    "learning_rate": learning_rate_val,
    "dropout_rate": dropout_val,
    "l1": l1_val,
    "l2": l2_val,
    "train_loss": train_loss,
    "val_loss": val_loss,
    # "train_accuracy": train_acc,
    # "val_accuracy": val_acc,
    "training_time_min": training_time_sec / 60.0
}

# Overall accuracy
overall_acc = np.mean(y_pred == y_val_bin)
row["accuracy"] = overall_acc  # add this to the CSV row

for i, cls in enumerate(CLASS_NAMES):
    cls_l = cls.lower()
    row[f"acc_{cls_l}"] = per_class_acc[i]
    row[f"precision_{cls_l}"] = report_dict[cls]["precision"]
    row[f"recall_{cls_l}"] = report_dict[cls]["recall"]
    row[f"f1_{cls_l}"] = report_dict[cls]["f1-score"]

file_exists = os.path.isfile(CSV_FILE)
with open(CSV_FILE, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Experiment {exp_id} logged to {CSV_FILE}")

# Save loss plot
plt.figure(figsize=(8, 6))
plt.plot(history.history["loss"], label="Train Loss", marker='o')
plt.plot(history.history["val_loss"], label="Val Loss", marker='o')
# plt.title(f"Experiment {exp_id:05d} Loss Curve")
if "accuracy" in history.history:
    plt.plot(history.history["accuracy"], label="Train Acc", marker="x")
    plt.plot(history.history["val_accuracy"], label="Val Acc", marker="x")
plt.title(f"Training Curves - Experiment {exp_id:05d}")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
# plot_filename = os.path.join(LOSS_PLOT_FOLDER, f"{exp_id:05d}.png")
# plt.savefig(plot_filename)
plt.savefig(os.path.join(CURVE_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()
# print(f"Loss plot saved to {plot_filename}")


fpr, tpr, thresholds = roc_curve(y_val_bin, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - Experiment {exp_id:05d}")
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(os.path.join(ROC_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()

Experiment 27 logged to experiments/binary_resnet50_roc_auc/roc_auc_binary_experiments.csv


In [20]:
# No longer required as threshold chosen as 0.595

# Ground-truth binary labels (0=Normal, 1=Abnormal)
y_true = y_val_bin

# ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)

# Youden's J statistic
youden_j = tpr - fpr

best_idx = np.argmax(youden_j)
best_threshold = thresholds[best_idx]

print(f"Best threshold (Youden's J): {best_threshold:.4f}")
print(f"Sensitivity (TPR): {tpr[best_idx]:.4f}")
print(f"Specificity (TNR): {1 - fpr[best_idx]:.4f}")


Best threshold (Youden's J): 0.7116
Sensitivity (TPR): 0.7025
Specificity (TNR): 0.6722


In [21]:
# No longer required as threshold chosen as 0.595
y_pred_opt = (y_pred_probs >= best_threshold).astype(int)

In [22]:
# No longer required as threshold chosen as 0.595
cm = confusion_matrix(y_true, y_pred_opt)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

print("Confusion matrix:\n", cm)
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

print(
    "\nClassification report:\n",
    classification_report(
        y_true,
        y_pred_opt,
        target_names=CLASS_NAMES
    )
)

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")

Confusion matrix:
 [[4502 2195]
 [ 293  692]]

Normalized confusion matrix:
 [[0.67 0.33]
 [0.3  0.7 ]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.94      0.67      0.78      6697
    Abnormal       0.24      0.70      0.36       985

    accuracy                           0.68      7682
   macro avg       0.59      0.69      0.57      7682
weighted avg       0.85      0.68      0.73      7682

Accuracy for class 0 (Normal): 0.6722
Accuracy for class 1 (Abnormal): 0.7025


In [23]:
# No longer required as threshold chosen as 0.595

thresholds = np.linspace(0.0, 1.0, 201)  # finer grid: step = 0.005
rows = []

for t in thresholds:
    y_pred = (y_pred_probs >= t).astype(int)
    cm = confusion_matrix(y_val_bin, y_pred)
    
    # Handle degenerate cases
    if cm.shape != (2, 2):
        continue

    tn, fp, fn, tp = cm.ravel()

    # Per-class accuracy
    acc_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
    acc_abnormal = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Overall metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision_abn = precision_score(y_val_bin, y_pred, pos_label=1, zero_division=0)
    recall_abn = recall_score(y_val_bin, y_pred, pos_label=1, zero_division=0)
    f1_abn = f1_score(y_val_bin, y_pred, pos_label=1, zero_division=0)

    rows.append({
        "threshold": t,
        "accuracy_overall": accuracy,
        "acc_normal": acc_normal,
        "acc_abnormal": acc_abnormal,
        "precision_abnormal": precision_abn,
        "recall_abnormal": recall_abn,
        "f1_abnormal": f1_abn,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    })

df = pd.DataFrame(rows)

# --- Apply your constraints ---
filtered = df[
    (df["acc_normal"] >= 0.6) &
    (df["acc_abnormal"] >= 0.7)
].copy()

print(f"Thresholds satisfying constraints: {len(filtered)}")

# Sort by what matters most to you (example: recall first, then F1)
filtered = filtered.sort_values(
    by=["recall_abnormal", "f1_abnormal"],
    ascending=False
)

pd.set_option("display.float_format", "{:.3f}".format)
print(filtered.head(10))


Thresholds satisfying constraints: 34
     threshold  accuracy_overall  acc_normal  acc_abnormal  \
109      0.545             0.622       0.601         0.766   
110      0.550             0.624       0.603         0.765   
111      0.555             0.625       0.605         0.764   
112      0.560             0.627       0.607         0.761   
113      0.565             0.628       0.609         0.760   
114      0.570             0.630       0.611         0.759   
115      0.575             0.631       0.613         0.756   
116      0.580             0.632       0.614         0.754   
117      0.585             0.634       0.617         0.750   
118      0.590             0.634       0.618         0.747   

     precision_abnormal  recall_abnormal  f1_abnormal    tn    fp   fn   tp  
109               0.220            0.766        0.342  4025  2672  230  755  
110               0.221            0.765        0.343  4036  2661  231  754  
111               0.221            0.764     

In [24]:
y_pred_opt_selected = (y_pred_probs >= 0.595).astype(int)

In [25]:
cm = confusion_matrix(y_true, y_pred_opt_selected)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

print("Confusion matrix:\n", cm)
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

print(
    "\nClassification report:\n",
    classification_report(
        y_true,
        y_pred_opt,
        target_names=CLASS_NAMES
    )
)

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")

Confusion matrix:
 [[4151 2546]
 [ 253  732]]

Normalized confusion matrix:
 [[0.62 0.38]
 [0.26 0.74]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.94      0.67      0.78      6697
    Abnormal       0.24      0.70      0.36       985

    accuracy                           0.68      7682
   macro avg       0.59      0.69      0.57      7682
weighted avg       0.85      0.68      0.73      7682

Accuracy for class 0 (Normal): 0.6198
Accuracy for class 1 (Abnormal): 0.7431
